# Task 1 — Inconsistencies and duplicates


Student clubs collected sign-ups through an online form for one week. The goal is a clean table with
**one row per student per club**, every value spelled one way.

**Rules followed throughout**

1. **Bronze is never edited.** `club_signups.csv` is read into `raw` and never changed; every fix happens on a copy, `signups`.
2. **Every fix is code.** Nothing is edited by hand, so the notebook re-runs on next week's export.
3. **Count before and after.** Every step prints how many rows or values it changed.
4. **Assert the result.** Each step ends with a check that fails loudly if the data breaks the rule.

| Section | What happens |
|---|---|
| 0 · Look first | Shape, dtypes, `value_counts()`, the six questions, a profile of every column |
| A · Inconsistencies | faculty / club / city → canonical values; names and emails; `fee_paid` → boolean; `signed_up_at` → one datetime column |
| B · Duplicates | exact duplicates; the same sign-up submitted again (keep the latest); `student_id` + `club` unique |
| Save | `club_signups_clean.csv` |
| C · Evidence for the README | the numbers, the order-of-steps experiment, and the two students with the same name |

## Setup

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

RAW_FILE = Path("club_signups.csv")
CLEAN_FILE = Path("club_signups_clean.csv")

## 0 · Look first

Read every column as **text** (`dtype=str`), exactly as the students typed it. Otherwise pandas would guess
types per column, and a value like `"1"` in `fee_paid` or `"61-4844"` in `student_id` could be turned into
something else before we've even looked at it. `raw` is the bronze table: read all notebook, never changed.

In [2]:
raw = pd.read_csv(RAW_FILE, dtype=str)
print("shape:", raw.shape)  # (rows, columns)
raw.head()

shape: (39, 9)


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Soccer,2026-09-15 13:39,N
1,S010,61-3846,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,Management,alexandria,Chess,15/09/2026 03:12,false
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32,Y
3,S035,61-4844,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,Engineering and Materials Science,Cairo,Debating,2026-09-17 11:15,yes
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess Club,2026-09-13 22:36,yes


In [3]:
raw.dtypes  # every column is text (object): the types are fixed in Part A

signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object

### `value_counts()` of every categorical column

The problems are visible here, before anything is fixed.

In [4]:
CATEGORICAL = ["faculty", "city", "club", "fee_paid"]

for column in CATEGORICAL:
    counts = raw[column].value_counts(dropna=False)
    print(f"--- {column}: {counts.size} distinct values")
    print(counts.to_string(), end="\n\n")

--- faculty: 13 distinct values
faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1

--- city: 9 distinct values
city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO          1
El Giza        1

--- club: 13 distinct values
club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music          2
chess        

- **faculty:** 13 spellings of 5 faculties: abbreviations (`IET`), full names (`Information Engineering and Technology`), case (`iet`), and a short form (`Pharma`).
- **city:** 9 spellings of 3 cities: case (`CAIRO`), and the aliases `Alex` and `El Giza`.
- **club:** 13 spellings of 5 clubs: case, a trailing space (`"Music "`), a `Club` suffix (`Chess Club`), `Debating`/`debate club` for Debate, and **`Soccer` for Football**.
- **fee_paid:** 9 ways of writing yes/no: `yes`, `Y`, `TRUE`, `1`, `no`, `N`, `false`, `0`…

### The six questions

Asked of every new dataset before fixing anything.

In [5]:
# 1 · What is one row (the grain)?  One FORM SUBMISSION. The target grain is one row per STUDENT per CLUB.
# 2 · Is the key unique?
print("signup_id unique?", raw["signup_id"].is_unique,
      f"-> {len(raw)} rows, {raw['signup_id'].nunique()} distinct signup_ids")
print("student_id + club unique?", not raw.duplicated(subset=["student_id", "club"]).any())

# 5 · Where are the nulls?
print("missing values per column:", raw.isna().sum().to_dict())

signup_id unique? False -> 39 rows, 36 distinct signup_ids
student_id + club unique? False
missing values per column: {'signup_id': 0, 'student_id': 0, 'full_name': 0, 'email': 0, 'faculty': 0, 'city': 0, 'club': 0, 'signed_up_at': 0, 'fee_paid': 0}


The profile function from Lab 3 asks the same questions of **every column at once**:

| Question | Where the profile answers it |
|---|---|
| 1 · What is one row? | `distinct` of `signup_id` vs the row count |
| 2 · Is the key unique? | `distinct` of `signup_id` (36 for 39 rows); `most common` shows ids that appear twice |
| 3 · Is each column the right type? | `dtype`: everything is text, including a timestamp and a yes/no column |
| 4 · What do categories look like? | `distinct` and `most common` of faculty / city / club / fee_paid |
| 5 · Where are the nulls? | `missing` |
| 6 · What are the ranges? | `min` / `max` and the text lengths; the date range is checked after parsing in A5 |

In [6]:
def profile(frame: pd.DataFrame) -> pd.DataFrame:
    # One row per column: its type, nulls, distinct values, range, and most common values.
    rows = {}
    for name, column in frame.items():
        lengths = column.dropna().str.len()
        rows[name] = {
            "dtype": str(column.dtype),
            "missing": column.isna().sum(),
            "distinct": column.nunique(),
            "min": column.min(),
            "max": column.max(),
            "text length": f"{lengths.min()}-{lengths.max()}",
            "most common": column.value_counts().head(3).to_dict(),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


profile(raw)

,dtype,missing,distinct,min,max,text length,most common
signup_id,object,0,36,S001,S036,4-4,"{'S015': 2, 'S022': 2, 'S023': 2}"
student_id,object,0,15,55-2639,61-8942,7-7,"{'61-7344': 6, '55-9245': 5, '61-4844': 4}"
full_name,object,0,24,Farida Ibrahim,mohamed adel,9-18,"{'Laila Mostafa': 4, 'Hana Zaki': 3, 'Sara Zak..."
email,object,0,18,mohamed.adel@student.guc.edu.eg,ziad.saleh@student.guc.edu.eg,28-33,"{'ali.ibrahim@student.guc.edu.eg': 5, 'mohamed..."
faculty,object,0,13,EMS,pharmacy,3-38,"{'Pharmacy': 7, 'EMS': 7, 'IET': 5}"
city,object,0,9,Alex,giza,4-10,"{'Cairo': 13, 'Alexandria': 11, 'Giza': 4}"
club,object,0,13,Chess,music,5-11,"{'Robotics': 8, 'Music': 5, 'Chess': 4}"
signed_up_at,object,0,36,13/09/2026 15:38,2026-09-17 23:36,16-16,"{'2026-09-15 13:39': 2, '2026-09-16 01:39': 2,..."
fee_paid,object,0,9,0,yes,1-5,"{'1': 7, 'yes': 7, 'false': 6}"


**What the look showed** (each item is fixed below):

1. **Spelling variants** in `faculty`, `city` and `club` (A2): 13, 9 and 13 distinct values for 5, 3 and 5 real ones.
2. **Names** with extra spaces (`"  Mohamed  Adel "`, text length up to 16) and mixed case (`AHMED ADEL`, `laila mostafa`); **emails** with spaces and upper case (A3).
3. **`fee_paid`** is text with 9 spellings, not a boolean (A4).
4. **`signed_up_at`** mixes two formats, `2026-09-15 13:39` and `15/09/2026 03:12` (A5).
5. **`signup_id` is not unique** (36 for 39 rows): some submissions appear twice (B1).
6. No missing values anywhere, so there is nothing to impute.

## Part A · Inconsistencies

The working copy. From here on `raw` is never touched.

In [7]:
signups = raw.copy()


def normalise(values: pd.Series) -> pd.Series:
    # Mechanical normalisation: strip the ends, collapse inner whitespace, lower-case.
    return values.str.strip().str.replace(r"\s+", " ", regex=True).str.lower()

### A2 · Faculty, club, city → canonical values

The recipe from Lab 3, for every categorical column:

1. **Look:** `value_counts()` (done above).
2. **Normalise mechanically:** strip, collapse spaces, lower-case. This fixes the case variants and `"Music "` at once.
3. **Map what's left** with an explicit dictionary: abbreviations, full names and aliases a normaliser can't fix.
   `.map(dict)` returns NaN for anything not in the dictionary, so `.fillna(key.str.title())` keeps the plain names (`"cairo"` → `"Cairo"`).
4. **Assert** that only the canonical values remain. If a new spelling shows up in next week's export, this fails instead of passing silently.

No fuzzy matching: every alias is listed and checked by a person.

In [8]:
FACULTIES = {"MET", "IET", "EMS", "Pharmacy", "Management"}
CLUBS = {"Robotics", "Debate", "Music", "Football", "Chess"}
CITIES = {"Cairo", "Giza", "Alexandria"}

FACULTY_ALIASES = {
    "met": "MET", "media engineering and technology": "MET",
    "iet": "IET", "information engineering and technology": "IET",
    "ems": "EMS", "engineering and materials science": "EMS",
    "pharma": "Pharmacy",
}
CLUB_ALIASES = {
    "chess club": "Chess",
    "debate club": "Debate", "debating": "Debate",
    "soccer": "Football",  # the Soccer club IS the Football club
}
CITY_ALIASES = {"alex": "Alexandria", "el giza": "Giza"}


def canonical(values: pd.Series, aliases: dict) -> pd.Series:
    key = normalise(values)
    return key.map(aliases).fillna(key.str.title())


signups["faculty"] = canonical(signups["faculty"], FACULTY_ALIASES)
signups["club"] = canonical(signups["club"], CLUB_ALIASES)
signups["city"] = canonical(signups["city"], CITY_ALIASES)

In [9]:
# ✓ check: only the canonical values remain
assert set(signups["faculty"]) == FACULTIES, set(signups["faculty"]) - FACULTIES
assert set(signups["club"]) == CLUBS, set(signups["club"]) - CLUBS
assert set(signups["city"]) == CITIES, set(signups["city"]) - CITIES

for column in ["faculty", "club", "city"]:
    print(f"{column}: {raw[column].nunique()} spellings -> {signups[column].nunique()}:",
          signups[column].value_counts().to_dict())

faculty: 13 spellings -> 5: {'IET': 12, 'Pharmacy': 9, 'EMS': 9, 'MET': 5, 'Management': 4}
club: 13 spellings -> 5: {'Football': 9, 'Chess': 9, 'Music': 8, 'Robotics': 8, 'Debate': 5}
city: 9 spellings -> 3: {'Alexandria': 16, 'Cairo': 16, 'Giza': 7}


### A3 · Names and emails

- **Names:** `.str.split()` with no argument splits on any run of whitespace and ignores the ends, so
  `.str.join(" ")` rebuilds the name with exactly one space between words. Then `.str.title()` gives everyone
  the same capitalisation: `"  Mohamed  Adel "`, `"MOHAMED ADEL"` and `"mohamed adel"` all become `"Mohamed Adel"`.
- **Emails:** trimmed and lower-case.

In [10]:
signups["full_name"] = signups["full_name"].str.split().str.join(" ").str.title()
signups["email"] = signups["email"].str.strip().str.lower()

changed_names = (raw["full_name"] != signups["full_name"]).sum()
changed_emails = (raw["email"] != signups["email"]).sum()
print(f"names changed: {changed_names}   emails changed: {changed_emails}")

pd.DataFrame({"name as typed": raw["full_name"], "name": signups["full_name"],
              "email as typed": raw["email"], "email": signups["email"]})[
    (raw["full_name"] != signups["full_name"]) | (raw["email"] != signups["email"])
].head(8)

names changed: 14   emails changed: 5


,name as typed,name,email as typed,email
1,AHMED ZAKI,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,ahmed.zaki@student.guc.edu.eg
2,Mohamed Adel,Mohamed Adel,mohamed.adel@student.guc.edu.eg,mohamed.adel@student.guc.edu.eg
3,MOHAMED ADEL,Mohamed Adel,mohamed.adel@student.guc.edu.eg,mohamed.adel@student.guc.edu.eg
5,ALI IBRAHIM,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,ali.ibrahim@student.guc.edu.eg
6,ahmed adel,Ahmed Adel,AHMED.ADEL@STUDENT.GUC.EDU.EG,ahmed.adel@student.guc.edu.eg
7,OMAR ADEL,Omar Adel,omar.adel@student.guc.edu.eg,omar.adel@student.guc.edu.eg
8,ali ibrahim,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,ali.ibrahim@student.guc.edu.eg
9,ali ibrahim,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,ali.ibrahim@student.guc.edu.eg


In [11]:
# ✓ check: one spelling per person
assert not signups["full_name"].str.contains(r"^\s|\s$|\s\s", regex=True).any(), "extra spaces left in a name"
assert signups["full_name"].eq(signups["full_name"].str.title()).all(), "inconsistent capitalisation"
assert signups["email"].str.fullmatch(r"[a-z]+\.[a-z]+@student\.guc\.edu\.eg").all(), "an email has another shape"

# Each student_id now has exactly one name and one email: the id really identifies a person.
per_student = signups.groupby("student_id")[["full_name", "email"]].nunique()
assert (per_student == 1).all().all(), per_student[(per_student > 1).any(axis=1)]
print("✓", signups["student_id"].nunique(), "students, each with one name and one email")

✓ 15 students, each with one name and one email


### A4 · `fee_paid` → a boolean column

Normalise, then map every spelling with an explicit dictionary. `.map` gives NaN for any spelling **not** in
the dictionary, so asserting there are no NaNs means no value was silently guessed.

In [12]:
FEE_PAID = {
    "yes": True, "y": True, "true": True, "1": True,
    "no": False, "n": False, "false": False, "0": False,
}

fee = normalise(signups["fee_paid"]).map(FEE_PAID)
assert fee.notna().all(), f"unknown spellings: {set(signups.loc[fee.isna(), 'fee_paid'])}"
signups["fee_paid"] = fee.astype(bool)

# ✓ check
assert signups["fee_paid"].dtype == bool
signups["fee_paid"].value_counts().to_dict()

{True: 27, False: 12}

### A5 · `signed_up_at` → one datetime column

First label each row with its format, then answer the questions **from the data**, not by guessing.

In [13]:
is_slash = signups["signed_up_at"].str.contains("/", regex=False)
fmt = is_slash.map({False: "YYYY-MM-DD HH:MM", True: "??/??/YYYY HH:MM"})
signups.groupby(fmt)["signed_up_at"].agg(rows="size", example="first")

,rows,example
signed_up_at,,
??/??/YYYY HH:MM,14,15/09/2026 03:12
YYYY-MM-DD HH:MM,25,2026-09-15 13:39


In [14]:
slashed = signups.loc[is_slash, "signed_up_at"]
first, second = slashed.str[:2].astype(int), slashed.str[3:5].astype(int)
print("first part  > 12:", (first > 12).sum(), "rows, values", sorted(first.unique().tolist()))
print("second part > 12:", (second > 12).sum(), "rows, values", sorted(second.unique().tolist()))

first part  > 12: 14 rows, values [13, 14, 15, 16, 17, 18]
second part > 12: 0 rows, values [9]


- The first part goes above 12 (13–18) and can't be a month. The second part is always `09`. So the slash format
  is **day/month/year**: `15/09/2026` is 15 September. (Read month-first, `15/09` would be month 15, which is impossible.)
- The other format, `2026-09-15 13:39`, is ISO: year-month-day, no ambiguity.
- Both formats give dates in the same week of September 2026, which is the one week the form was open.
- Neither format carries a timezone, so the column stays **local time** (the form's time in Cairo).

Parse each format with an **explicit** `format=` (never let pandas guess day vs month), into one column:

In [15]:
parsed = pd.Series(pd.NaT, index=signups.index, dtype="datetime64[ns]")  #Make an EMPTY datetime column to fill in , pd.NaT = "Not a Time"
parsed[~is_slash] = pd.to_datetime(signups.loc[~is_slash, "signed_up_at"], format="%Y-%m-%d %H:%M")
parsed[is_slash] = pd.to_datetime(signups.loc[is_slash, "signed_up_at"], format="%d/%m/%Y %H:%M")
signups["signed_up_at"] = parsed    # Replace the old text column with this single datetime column.

# ✓ check
assert signups["signed_up_at"].notna().all(), "a timestamp didn't parse"
assert str(signups["signed_up_at"].dtype) == "datetime64[ns]"
s010 = signups.loc[signups["signup_id"] == "S010", "signed_up_at"].iloc[0]  # typed as "15/09/2026 03:12"
assert s010 == pd.Timestamp("2026-09-15 03:12"), s010
print("✓ one datetime column, from", signups["signed_up_at"].min(), "to", signups["signed_up_at"].max())

✓ one datetime column, from 2026-09-13 14:36:00 to 2026-09-18 23:36:00


In [16]:
signups.dtypes  # Part A done: every column has one type, every categorical one spelling


signup_id               object
student_id              object
full_name               object
email                   object
faculty                 object
city                    object
club                    object
signed_up_at    datetime64[ns]
fee_paid                  bool
dtype: object

## Part B · Duplicates

| Kind | Looks like | Fix |
|---|---|---|
| **Exact** | The same form submission twice: every column identical, including `signup_id` | Drop the extra copies |
| **Same key, different content** | Same `student_id` + `club`, a different `signup_id` | Keep the **latest** submission |
| **Same name, different student** | Two students called *Mohamed Adel* | **Not** a duplicate: the key is `student_id`, never the name |

### B1 · Exact duplicates

In [17]:
rows_start = len(signups)
exact = signups.duplicated().sum()
print(f"exact duplicate rows: {exact}")
signups[signups.duplicated(keep=False)].sort_values("signup_id")  # every copy(all copies not just the extras), side by side

exact duplicate rows: 3


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
34,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
15,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
24,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
20,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
33,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False


In [18]:
signups = signups.drop_duplicates().reset_index(drop=True)
rows_after_exact = len(signups)

# ✓ check: every form submission now appears once
assert signups["signup_id"].is_unique
print(f"✓ exact duplicates removed: {rows_start} -> {rows_after_exact} rows ({rows_start - rows_after_exact} removed)")

✓ exact duplicates removed: 39 -> 36 rows (3 removed)


### B2 · The same sign-up submitted again

Same `student_id` and `club`, but a different `signup_id`: the student filled in the form a second time.

In [19]:
KEY = ["student_id", "club"]
repeated = signups[signups.duplicated(subset=KEY, keep=False)].sort_values(KEY + ["signed_up_at"])
print(f"{repeated.groupby(KEY).ngroups} student/club pairs were submitted more than once:")
repeated[["signup_id", "student_id", "full_name", "club", "signed_up_at", "fee_paid"]]

4 student/club pairs were submitted more than once:


,signup_id,student_id,full_name,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,Music,2026-09-16 09:07:00,True
22,S021,61-4844,Mohamed Adel,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,Debate,2026-09-17 11:15:00,True
32,S017,61-7344,Laila Mostafa,Debate,2026-09-15 18:01:00,False
24,S034,61-7344,Laila Mostafa,Debate,2026-09-16 02:01:00,True
21,S032,61-7344,Laila Mostafa,Football,2026-09-17 23:36:00,False
35,S036,61-7344,Laila Mostafa,Football,2026-09-18 23:36:00,True


**Which copy wins? The latest.** Each repeat is a newer version of the same sign-up. In all 4 pairs the
student came back to mark the fee as **paid** (`False` → `True`). Keeping the first copy would record them as
unpaid and send fee reminders to students who already paid. "Keep the latest" is a business rule, not a pandas
default: `drop_duplicates` keeps the *first* row unless told otherwise.

"Latest" is decided by `signed_up_at`, the time of the submission, not by the order of rows in the file.
That's why the column had to be a real datetime first.

In [20]:
# the rule only works if "latest" is unambiguous: no two copies of a pair share a timestamp
assert not signups.duplicated(subset=KEY + ["signed_up_at"]).any()

before = len(signups)
signups = (
    signups.sort_values("signed_up_at")                   # oldest first ...
    .drop_duplicates(subset=KEY, keep="last")             # ... so "last" is the latest submission
    .sort_values("signup_id")
    .reset_index(drop=True)
)
rows_after_resubmit = len(signups)
removed = sorted(set(repeated["signup_id"]) - set(signups["signup_id"]))
print(f"rows removed: {before - rows_after_resubmit}  ({before} -> {rows_after_resubmit})   dropped: {removed}")

rows removed: 4  (36 -> 32)   dropped: ['S017', 'S020', 'S021', 'S032']


### B3 · Check: one row per student per club

In [21]:
assert not signups.duplicated(subset=KEY).any(), "student_id + club is not unique"
assert signups.set_index(KEY).index.is_unique
print(f"✓ student_id + club is unique: {len(signups)} rows, {signups['student_id'].nunique()} students")

✓ student_id + club is unique: 32 rows, 15 students


### Final checks on the clean table

The promises the clean file makes, all in one place.

In [22]:
checks = pd.Series({
    "faculty is canonical": signups["faculty"].isin(FACULTIES).all(),
    "club is canonical": signups["club"].isin(CLUBS).all(),
    "city is canonical": signups["city"].isin(CITIES).all(),
    "fee_paid is boolean": signups["fee_paid"].dtype == bool,
    "signed_up_at is one datetime column": str(signups["signed_up_at"].dtype) == "datetime64[ns]",
    "names: one spelling, no extra spaces": signups["full_name"].eq(signups["full_name"].str.split().str.join(" ").str.title()).all(),
    "emails trimmed and lower-case": signups["email"].eq(signups["email"].str.strip().str.lower()).all(),
    "signup_id unique": signups["signup_id"].is_unique,
    "student_id + club unique": not signups.duplicated(subset=KEY).any(),
    "no missing values": signups.notna().all().all(),
})
assert checks.all(), checks[~checks]
checks

faculty is canonical                    True
club is canonical                       True
city is canonical                       True
fee_paid is boolean                     True
signed_up_at is one datetime column     True
names: one spelling, no extra spaces    True
emails trimmed and lower-case           True
signup_id unique                        True
student_id + club unique                True
no missing values                       True
dtype: bool

## Save

In [23]:
signups.to_csv(CLEAN_FILE, index=False)  # index=False: don't write 0..n-1 as an extra column

# read it back as a consumer would, and check it survived the round trip
check = pd.read_csv(CLEAN_FILE, parse_dates=["signed_up_at"], dtype={"student_id": str})
assert len(check) == len(signups) and check["fee_paid"].dtype == bool
assert not check.duplicated(subset=KEY).any()
print(f"✓ saved {CLEAN_FILE}: {len(check)} rows; {RAW_FILE} left unchanged")
check.head()

✓ saved club_signups_clean.csv: 32 rows; club_signups.csv left unchanged


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
2,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
3,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
4,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True


## Part C · Evidence for the README

### The numbers

In [24]:
print(f"started with:                     {rows_start} rows")
print(f"exact duplicates removed:         {rows_start - rows_after_exact}  -> {rows_after_exact}")
print(f"same sign-up resubmitted removed: {rows_after_exact - rows_after_resubmit}  -> {rows_after_resubmit}")
print(f"left:                             {rows_after_resubmit} rows (one per student per club)")

started with:                     39 rows
exact duplicates removed:         3  -> 36
same sign-up resubmitted removed: 4  -> 32
left:                             32 rows (one per student per club)


### Does the order of the steps matter?

Experiment: remove the duplicates **before** fixing the club spellings, on the raw text.

In [25]:
early = raw.drop_duplicates()                       # exact duplicates: identical either way
early_exact = len(raw) - len(early)
early_resubmits = early.duplicated(subset=KEY).sum()  # club spellings NOT fixed yet

print(f"exact duplicates found before the fixes: {early_exact} (after: {rows_start - rows_after_exact})")
print(f"resubmissions found before the fixes:    {early_resubmits} (after: {rows_after_exact - rows_after_resubmit})")

# the pairs the early run misses: the same club, typed two different ways
missed = early.assign(club_clean=canonical(early["club"], CLUB_ALIASES))
missed = missed[missed.duplicated(subset=["student_id", "club_clean"], keep=False)
                & ~missed.duplicated(subset=KEY, keep=False)]
missed.sort_values(["student_id", "club_clean"])[["signup_id", "student_id", "full_name", "club", "club_clean"]]

exact duplicates found before the fixes: 3 (after: 3)
resubmissions found before the fixes:    2 (after: 4)


,signup_id,student_id,full_name,club,club_clean
8,S020,55-9245,ali ibrahim,Music,Music
9,S033,55-9245,ali ibrahim,music,Music
25,S034,61-7344,Laila Mostafa,debate club,Debate
35,S017,61-7344,Laila Mostafa,Debate,Debate


In [26]:
# what the early run would leave behind
left_early = len(early) - early_resubmits
still_duplicated = early.drop_duplicates(subset=KEY).assign(club=lambda d: canonical(d["club"], CLUB_ALIASES)).duplicated(subset=KEY).sum()
print(f"rows left: {left_early} instead of {rows_after_resubmit};",
      f"{still_duplicated} student/club pairs would still appear twice, and the B3 assertion would fail")

rows left: 34 instead of 32; 2 student/club pairs would still appear twice, and the B3 assertion would fail


Exact duplicates are found either way: they are byte-for-byte copies of one submission. The resubmissions are
not, because a student may type the club differently the second time (`Music`/`music`, `Debate`/`debate club`).
Until the spellings are fixed, those look like two different clubs. **Consistency first, then key-based
de-duplication.**

### Two different students with the same name

In [27]:
same_name = signups.groupby("full_name")["student_id"].nunique().loc[lambda n: n > 1]
print("names shared by more than one student_id:", same_name.to_dict())
signups[signups["full_name"].isin(same_name.index)].sort_values(["student_id", "club"])[
    ["signup_id", "student_id", "full_name", "email", "faculty", "city", "club"]
]

names shared by more than one student_id: {'Mohamed Adel': 2}


,signup_id,student_id,full_name,email,faculty,city,club
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate
20,S024,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Robotics
30,S035,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate
7,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music
8,S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics


In [28]:
# What de-duplicating on the NAME (or the email) instead of student_id would have done:
by_name = signups.duplicated(subset=["full_name", "club"]).sum()
by_email = signups.duplicated(subset=["email", "club"]).sum()
print(f"extra rows a name-based rule would delete:  {by_name}")
print(f"extra rows an email-based rule would delete: {by_email}")

extra rows a name-based rule would delete:  2
extra rows an email-based rule would delete: 2


Two different students are called **Mohamed Adel** (`61-4844`, EMS, Cairo and `55-2992`, IET, Alexandria). They
even typed the **same email**. Both signed up for **Debate** and **Robotics**, so a rule keyed on the name, or on
the email, would treat those as resubmissions and delete 2 real sign-ups. Every duplicate rule here uses
**`student_id`**, the only column that identifies a person, so the two are never merged. A3 also checked that
each `student_id` has exactly one name and one email.